In [3]:
# %%
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path(r"D:\TechTrech\Tasks\Final Project\industrial_maintenance.db")
conn = sqlite3.connect(DB_PATH)

# Increase display so nothing is truncated
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

queries = {
"Q1 - Basic JOIN 3 tables": """
SELECT m.machine_id, m.product_id, m.type, s.timestamp, s.torque_nm, COALESCE(f.machine_failure,0) AS failed, COALESCE(f.failure_type,'NONE') AS failure_type
FROM machines m
JOIN sensor_readings s ON m.machine_id = s.machine_id
LEFT JOIN failure_events f ON s.reading_id = f.reading_id
ORDER BY s.timestamp DESC LIMIT 10
""",
"Q2 - Failure ratio per Type": """
SELECT m.type, COUNT(*) AS total_readings, SUM(COALESCE(f.machine_failure,0)) AS total_failures,
ROUND(100.0 * SUM(COALESCE(f.machine_failure,0)) / COUNT(*), 2) AS failure_rate_pct
FROM machines m
JOIN sensor_readings s ON m.machine_id = s.machine_id
LEFT JOIN failure_events f ON s.reading_id = f.reading_id
GROUP BY m.type ORDER BY failure_rate_pct DESC
""",
"Q3 - CTE failure rate by type": """
WITH type_stats AS (
    SELECT m.type, COUNT(*) AS total, SUM(COALESCE(f.machine_failure,0)) AS failures
    FROM machines m JOIN sensor_readings s ON m.machine_id = s.machine_id LEFT JOIN failure_events f ON s.reading_id = f.reading_id GROUP BY m.type
)
SELECT type, total, failures, ROUND(100.0 * failures / total,2) AS failure_rate_pct FROM type_stats ORDER BY failure_rate_pct DESC
""",
"Q4 - Avg Torque failure vs normal": """
SELECT CASE WHEN COALESCE(f.machine_failure,0)=1 THEN 'FAILURE' ELSE 'NORMAL' END AS status,
COUNT(*) AS count, ROUND(AVG(s.torque_nm),2) AS avg_torque, ROUND(AVG(s.rotational_speed_rpm),2) AS avg_rpm, ROUND(AVG(s.tool_wear_min),2) AS avg_wear
FROM sensor_readings s LEFT JOIN failure_events f ON s.reading_id = f.reading_id GROUP BY status
""",
"Q5 - RANK top machines by wear": """
SELECT m.machine_id, m.product_id, m.type, s.tool_wear_min, COALESCE(f.machine_failure,0) AS failed,
RANK() OVER (ORDER BY s.tool_wear_min DESC) AS wear_rank
FROM machines m JOIN sensor_readings s ON m.machine_id = s.machine_id LEFT JOIN failure_events f ON s.reading_id = f.reading_id
ORDER BY wear_rank LIMIT 10
""",
"Q6 - Moving avg torque AVG OVER": """
SELECT s.reading_id, s.timestamp, s.torque_nm,
ROUND(AVG(s.torque_nm) OVER (ORDER BY s.timestamp ROWS BETWEEN 9 PRECEDING AND CURRENT ROW),2) AS torque_10_moving_avg
FROM sensor_readings s ORDER BY s.timestamp LIMIT 10
""",
"Q7 - LAG temp before failure": """
WITH ordered AS (
    SELECT s.reading_id, s.timestamp, s.air_temperature_k, s.temp_diff_k, COALESCE(f.machine_failure,0) AS machine_failure,
    LAG(s.air_temperature_k,1) OVER (ORDER BY s.timestamp) AS prev_air_temp
    FROM sensor_readings s LEFT JOIN failure_events f ON s.reading_id = f.reading_id
)
SELECT reading_id, timestamp, air_temperature_k, prev_air_temp, ROUND(air_temperature_k - prev_air_temp,2) AS delta, machine_failure
FROM ordered WHERE machine_failure=1 ORDER BY timestamp LIMIT 10
""",
"Q8 - Failure mode distribution": """
SELECT f.failure_type, COUNT(*) AS occurrences, ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM failure_events),2) AS pct,
ROUND(AVG(s.tool_wear_min),1) AS avg_wear
FROM failure_events f JOIN sensor_readings s ON f.reading_id = s.reading_id GROUP BY f.failure_type ORDER BY occurrences DESC
""",
"Q9 - Tool wear bins": """
SELECT CASE WHEN s.tool_wear_min <50 THEN '0-50' WHEN s.tool_wear_min <100 THEN '50-100' WHEN s.tool_wear_min <150 THEN '100-150' WHEN s.tool_wear_min <200 THEN '150-200' ELSE '200+' END AS wear_bin,
COUNT(*) AS total, SUM(COALESCE(f.machine_failure,0)) AS failures, ROUND(100.0 * SUM(COALESCE(f.machine_failure,0)) / COUNT(*),2) AS failure_rate
FROM sensor_readings s LEFT JOIN failure_events f ON s.reading_id = f.reading_id GROUP BY wear_bin ORDER BY MIN(s.tool_wear_min)
""",
"Q10 - Power PWF analysis": """
SELECT CASE WHEN s.power_w <3500 THEN 'LOW' WHEN s.power_w >9000 THEN 'HIGH' ELSE 'NORMAL' END AS power_cat,
COUNT(*) AS cnt, SUM(COALESCE(f.machine_failure,0)) AS failures, SUM(COALESCE(f.pwf,0)) AS pwf
FROM sensor_readings s LEFT JOIN failure_events f ON s.reading_id = f.reading_id GROUP BY power_cat ORDER BY failures DESC
""",
"Q11 - HDF condition": """
SELECT CASE WHEN s.temp_diff_k <8.6 AND s.rotational_speed_rpm <1380 THEN 'HDF_RISK' ELSE 'NORMAL' END AS cond,
COUNT(*) AS total, SUM(COALESCE(f.hdf,0)) AS hdf_failures, ROUND(100.0 * SUM(COALESCE(f.hdf,0)) / COUNT(*),2) AS hdf_rate
FROM sensor_readings s LEFT JOIN failure_events f ON s.reading_id = f.reading_id GROUP BY cond
""",
"Q12 - Multiple failure modes": """
SELECT f.machine_id, GROUP_CONCAT(f.failure_type, ',') AS types, COUNT(*) AS num_modes
FROM failure_events f GROUP BY f.reading_id HAVING COUNT(*)>1 LIMIT 10
""",
"Q13 - Monthly trend with window": """
WITH monthly AS (
    SELECT strftime('%Y-%m', s.timestamp) AS month, COUNT(*) AS readings, SUM(COALESCE(f.machine_failure,0)) AS failures
    FROM sensor_readings s LEFT JOIN failure_events f ON s.reading_id = f.reading_id GROUP BY month
)
SELECT month, readings, failures, ROUND(100.0 * failures / readings,2) AS rate,
ROUND(AVG(failures) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),2) AS moving_avg
FROM monthly ORDER BY month LIMIT 12
""",
"Q14 - Risk score top machines": """
WITH risk AS (
    SELECT m.machine_id, m.product_id, m.type, s.tool_wear_min, s.torque_nm, s.power_w, s.temp_diff_k,
    (CASE WHEN s.tool_wear_min >200 THEN 3 ELSE s.tool_wear_min/100.0 END + CASE WHEN s.power_w <3500 OR s.power_w >9000 THEN 2 ELSE 0 END + CASE WHEN s.temp_diff_k <8.6 AND s.rotational_speed_rpm <1380 THEN 2 ELSE 0 END + (s.tool_wear_min * s.torque_nm /10000.0)) AS risk_score
    FROM machines m JOIN sensor_readings s ON m.machine_id = s.machine_id
)
SELECT machine_id, product_id, type, tool_wear_min, ROUND(risk_score,2) AS risk_score, RANK() OVER (ORDER BY risk_score DESC) AS rank
FROM risk ORDER BY risk_score DESC LIMIT 10
"""
}

# Open file to save all outputs together
with open("all_queries_output.txt", "w", encoding="utf-8") as out_file:
    for name, sql in queries.items():
        try:
            print(f"\n{'='*70}", flush=True)
            print(f"{name}", flush=True)
            print(f"{'='*70}", flush=True)
            out_file.write(f"\n{'='*70}\n{name}\n{'='*70}\n")
            
            df = pd.read_sql_query(sql, conn)
            print(df.to_string(index=False), flush=True)
            out_file.write(df.to_string(index=False) + "\n")
            
            # Save individual CSV
            safe_name = name.split(' - ')[0]
            df.to_csv(f"{safe_name}_output.csv", index=False)
            
        except Exception as e:
            print(f"FAILED: {e}", flush=True)
            out_file.write(f"FAILED: {e}\n")

conn.close()
print("\n\nDONE! All 14 printed and saved to all_queries_output.txt + individual CSVs")


Q1 - Basic JOIN 3 tables
 machine_id product_id type           timestamp  torque_nm  failed failure_type
      10000     M24859    M 2020-02-04 17:20:00       40.2       0         NONE
       9999     H39412    H 2020-02-04 17:15:00       48.5       0         NONE
       9998     M24857    M 2020-02-04 17:10:00       33.4       0         NONE
       9997     H39410    H 2020-02-04 17:05:00       31.8       0         NONE
       9996     M24855    M 2020-02-04 17:00:00       29.5       0         NONE
       9995     L57174    L 2020-02-04 16:55:00       27.9       0         NONE
       9994     L57173    L 2020-02-04 16:50:00       47.3       0         NONE
       9993     L57172    L 2020-02-04 16:45:00       39.2       0         NONE
       9992     M24851    M 2020-02-04 16:40:00       26.1       0         NONE
       9991     L57170    L 2020-02-04 16:35:00       36.2       0         NONE

Q2 - Failure ratio per Type
type  total_readings  total_failures  failure_rate_pct
   L      